In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/hr_catalog/hr_core/hr_volume/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/DEI_Program_Outline_2025-01.pdf,DEI_Program_Outline_2025-01.pdf,1865,1788524004000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Exit_Process_Checklist_2025-01.pdf,Exit_Process_Checklist_2025-01.pdf,2015,1788524004000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Flexible_Work_Toolkit_2025-01.pdf,Flexible_Work_Toolkit_2025-01.pdf,1855,1788524005000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/GDPR_Employee_Data_Practices_2025-01.pdf,GDPR_Employee_Data_Practices_2025-01.pdf,2028,1788524004000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/HR_Quarterly_Summary_2025Q1.pdf,HR_Quarterly_Summary_2025Q1.pdf,1914,1788524005000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Performance_Calibration_Guide_2025-01.pdf,Performance_Calibration_Guide_2025-01.pdf,1872,1788524004000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Security_Awareness_Brief_2025-01.pdf,Security_Awareness_Brief_2025-01.pdf,2029,1788524004000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Travel_Policy_Quick_Reference_2025-01.pdf,Travel_Policy_Quick_Reference_2025-01.pdf,1914,1788524005000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/employees_export_2025-01.csv,employees_export_2025-01.csv,18042,1788441436000
dbfs:/Volumes/hr_catalog/hr_core/hr_volume/hr_helpdesk_tickets_2025-01.csv,hr_helpdesk_tickets_2025-01.csv,86616,1788441466000


In [0]:
from pyspark.sql import Row

# List all files in the volume
files = dbutils.fs.ls("/Volumes/hr_catalog/hr_core/hr_volume/")

# Filter for PDF files and extract name and path
pdf_files = [
    Row(file_name=file.name, file_path=file.path)
    for file in files
    if file.name.endswith('.pdf')
]

# Create DataFrame
df = spark.createDataFrame(pdf_files)

# Display preview
print(f"Found {df.count()} PDF files")
display(df)

# Create the table
df.write.mode("overwrite").saveAsTable("hr_catalog.hr_core.hr_documents")

print("\n✓ Table 'hr_catalog.hr_core.hr_documents' created successfully!")

Found 8 PDF files


file_name,file_path
DEI_Program_Outline_2025-01.pdf,dbfs:/Volumes/hr_catalog/hr_core/hr_volume/DEI_Program_Outline_2025-01.pdf
Exit_Process_Checklist_2025-01.pdf,dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Exit_Process_Checklist_2025-01.pdf
Flexible_Work_Toolkit_2025-01.pdf,dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Flexible_Work_Toolkit_2025-01.pdf
GDPR_Employee_Data_Practices_2025-01.pdf,dbfs:/Volumes/hr_catalog/hr_core/hr_volume/GDPR_Employee_Data_Practices_2025-01.pdf
HR_Quarterly_Summary_2025Q1.pdf,dbfs:/Volumes/hr_catalog/hr_core/hr_volume/HR_Quarterly_Summary_2025Q1.pdf
Performance_Calibration_Guide_2025-01.pdf,dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Performance_Calibration_Guide_2025-01.pdf
Security_Awareness_Brief_2025-01.pdf,dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Security_Awareness_Brief_2025-01.pdf
Travel_Policy_Quick_Reference_2025-01.pdf,dbfs:/Volumes/hr_catalog/hr_core/hr_volume/Travel_Policy_Quick_Reference_2025-01.pdf



✓ Table 'hr_catalog.hr_core.hr_documents' created successfully!


In [0]:
# Install LangChain if not already available
%pip install langchain langchain-text-splitters --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%sql
-- Parse PDFs using ai_parse_document
CREATE OR REPLACE TEMPORARY VIEW parsed_pdfs AS
SELECT
  _metadata.file_name AS document_name,
  ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
FROM READ_FILES(
  '/Volumes/hr_catalog/hr_core/hr_volume/',
  format => 'binaryFile'
)
WHERE _metadata.file_name LIKE '%.pdf';

-- Extract text from parsed documents
CREATE OR REPLACE TEMPORARY VIEW document_texts AS
SELECT
  document_name,
  concat_ws('\n\n',
    transform(
      try_cast(parsed_content:document:elements AS ARRAY<VARIANT>),
      element -> try_cast(element:content AS STRING)
    )
  ) AS full_text
FROM parsed_pdfs
WHERE is_variant_null(parsed_content:error_status);

SELECT 
  document_name,
  length(full_text) AS text_length,
  substring(full_text, 1, 200) AS text_preview
FROM document_texts;

document_name,text_length,text_preview
Security_Awareness_Brief_2025-01.pdf,227,"Security Awareness Brief 2025-01 Owner: marcin.dnbrowska@nordstar.example.com (IT) Version: 2025-01 • Top threats: phishing, MFA fatigue • Device encryption mandatory • Data handling Do’s & Don’ts •"
GDPR_Employee_Data_Practices_2025-01.pdf,215,GDPR Employee Data Practices 2025-01 Owner: noah.nowak@nordstar.example.com (HR) Version: 2025-01 • Lawful bases overview • DSAR intake ® response • Retention map snapshot • Vendors & DPAs • Cross-bo
Exit_Process_Checklist_2025-01.pdf,213,Exit Process Checklist 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 2025-01 • Notice ® Offboarding • Access revocation • Asset return • References & certificates • Post-terminat
HR_Quarterly_Summary_2025Q1.pdf,231,HR Quarterly Summary 2025Q1 Owner: sofia.wójcik@nordstar.example.com (HR) Version: 2025-01 • Headcount +2.3% QoQ • Turnover 8.1% (voluntary 61%) • L&D completion 86% • Pulse survey eNPS: +32 • Top th
Travel_Policy_Quick_Reference_2025-01.pdf,233,Travel Policy Quick Reference 2025-01 Owner: lucas.williams@nordstar.example.com (Finance) Version: 2025-01 • Hotel caps by city • Flights: Premium Econ >6h • Ride-shares reimbursable • Per diem vs r
Performance_Calibration_Guide_2025-01.pdf,200,Performance Calibration Guide 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 2025-01 • Rating distribution targets • Evidence-based discussions • Bias mitigations • Appeal process
DEI_Program_Outline_2025-01.pdf,196,DEI Program Outline 2025-01 Owner: marcin.smith@nordstar.example.com (HR) Version: 2025-01 • ERG launch plan • Inclusive leadership workshops • Hiring panels diversity • Quarterly metrics & goals
Flexible_Work_Toolkit_2025-01.pdf,182,Flexible Work Toolkit 2025-01 Owner: ava.white@nordstar.example.com (IT) Version: 2025-01 • Compressed weeks pilots • Overlap hours guidance • Asynchronous comms • Manager checklist


In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Read the parsed documents from the temp view
documents_df = spark.table("document_texts")
documents = documents_df.collect()

# Initialize LangChain text splitter with specified parameters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

# Process each document and create chunks
all_chunks = []
for doc in documents:
    document_name = doc.document_name
    full_text = doc.full_text
    
    # Split the text into chunks
    chunks = text_splitter.split_text(full_text)
    
    # Create a record for each chunk with metadata
    for chunk_id, chunk_text in enumerate(chunks, start=1):
        all_chunks.append(
            Row(
                document_name=document_name,
                chunk_id=chunk_id,
                chunk_text=chunk_text,
                chunk_length=len(chunk_text)
            )
        )

print(f"Created {len(all_chunks)} chunks from {len(documents)} documents")

# Create DataFrame with the chunks
chunks_df = spark.createDataFrame(all_chunks)

# Display sample of chunks
print("\nSample chunks:")
display(chunks_df.limit(5))

# Save to the table
chunks_df.write.mode("overwrite").saveAsTable("hr_catalog.hr_core.hr_document_chunks")

print(f"\n✓ Table 'hr_catalog.hr_core.hr_document_chunks' created with {len(all_chunks)} chunks!")

# Show summary statistics
print("\nSummary by document:")
chunks_df.groupBy("document_name").count().orderBy("document_name").show(truncate=False)

Created 8 chunks from 8 documents

Sample chunks:


document_name,chunk_id,chunk_text,chunk_length
Security_Awareness_Brief_2025-01.pdf,1,"Security Awareness Brief 2025-01 Owner: marcin.dnbrowska@nordstar.example.com (IT) Version: 2025-01 • Top threats: phishing, MFA fatigue • Device encryption mandatory • Data handling Do’s & Don’ts • Report incidents within 24h",227
GDPR_Employee_Data_Practices_2025-01.pdf,1,GDPR Employee Data Practices 2025-01 Owner: noah.nowak@nordstar.example.com (HR) Version: 2025-01 • Lawful bases overview • DSAR intake ® response • Retention map snapshot • Vendors & DPAs • Cross-border safeguards,215
Exit_Process_Checklist_2025-01.pdf,1,Exit Process Checklist 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 2025-01 • Notice ® Offboarding • Access revocation • Asset return • References & certificates • Post-termination reminders,213
HR_Quarterly_Summary_2025Q1.pdf,1,"HR Quarterly Summary 2025Q1 Owner: sofia.wójcik@nordstar.example.com (HR) Version: 2025-01 • Headcount +2.3% QoQ • Turnover 8.1% (voluntary 61%) • L&D completion 86% • Pulse survey eNPS: +32 • Top themes: career paths, flexibility",231
Travel_Policy_Quick_Reference_2025-01.pdf,1,Travel Policy Quick Reference 2025-01 Owner: lucas.williams@nordstar.example.com (Finance) Version: 2025-01 • Hotel caps by city • Flights: Premium Econ >6h • Ride-shares reimbursable • Per diem vs receipts • Pre-approval thresholds,233



✓ Table 'hr_catalog.hr_core.hr_document_chunks' created with 8 chunks!

Summary by document:
+-----------------------------------------+-----+
|document_name                            |count|
+-----------------------------------------+-----+
|DEI_Program_Outline_2025-01.pdf          |1    |
|Exit_Process_Checklist_2025-01.pdf       |1    |
|Flexible_Work_Toolkit_2025-01.pdf        |1    |
|GDPR_Employee_Data_Practices_2025-01.pdf |1    |
|HR_Quarterly_Summary_2025Q1.pdf          |1    |
|Performance_Calibration_Guide_2025-01.pdf|1    |
|Security_Awareness_Brief_2025-01.pdf     |1    |
|Travel_Policy_Quick_Reference_2025-01.pdf|1    |
+-----------------------------------------+-----+



In [0]:
%sql
-- Verify the final chunks table
SELECT 
  document_name,
  chunk_id,
  chunk_length,
  substring(chunk_text, 1, 100) AS chunk_preview
FROM hr_catalog.hr_core.hr_document_chunks
ORDER BY document_name, chunk_id;

document_name,chunk_id,chunk_length,chunk_preview
DEI_Program_Outline_2025-01.pdf,1,196,DEI Program Outline 2025-01 Owner: marcin.smith@nordstar.example.com (HR) Version: 2025-01 • ERG la
Exit_Process_Checklist_2025-01.pdf,1,213,Exit Process Checklist 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 2025-01 •
Flexible_Work_Toolkit_2025-01.pdf,1,182,Flexible Work Toolkit 2025-01 Owner: ava.white@nordstar.example.com (IT) Version: 2025-01 • Compres
GDPR_Employee_Data_Practices_2025-01.pdf,1,215,GDPR Employee Data Practices 2025-01 Owner: noah.nowak@nordstar.example.com (HR) Version: 2025-01 •
HR_Quarterly_Summary_2025Q1.pdf,1,231,HR Quarterly Summary 2025Q1 Owner: sofia.wójcik@nordstar.example.com (HR) Version: 2025-01 • Headco
Performance_Calibration_Guide_2025-01.pdf,1,200,Performance Calibration Guide 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 20
Security_Awareness_Brief_2025-01.pdf,1,227,Security Awareness Brief 2025-01 Owner: marcin.dnbrowska@nordstar.example.com (IT) Version: 2025-01
Travel_Policy_Quick_Reference_2025-01.pdf,1,233,Travel Policy Quick Reference 2025-01 Owner: lucas.williams@nordstar.example.com (Finance) Version:


In [0]:
import requests
import json
from pyspark.sql import Row

# Get authentication context
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()
host = ctx.apiUrl().get()

# Setup endpoint
url = f"{host}/serving-endpoints/databricks-bge-large-en/invocations"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

# Read chunks from the table
chunks_df = spark.table("hr_catalog.hr_core.hr_document_chunks")
chunks = chunks_df.collect()

print(f"Generating embeddings for {len(chunks)} chunks...\n")

# Generate embeddings for each chunk
embedding_rows = []
for i, chunk in enumerate(chunks, 1):
    try:
        # Call the embedding endpoint
        payload = {"input": [chunk.chunk_text]}
        response = requests.post(url, headers=headers, json=payload)
        
        if response.status_code == 200:
            result = response.json()
            embedding = result['data'][0]['embedding']
            
            # Create row with embedding
            embedding_rows.append(
                Row(
                    document_name=chunk.document_name,
                    chunk_id=chunk.chunk_id,
                    chunk_text=chunk.chunk_text,
                    embedding=embedding
                )
            )
            
            print(f"  ✓ Processed chunk {i}/{len(chunks)}: {chunk.document_name} (chunk {chunk.chunk_id})")
        else:
            print(f"  ✗ Error {response.status_code} for chunk {i}: {response.text[:100]}")
            
    except Exception as e:
        print(f"  ✗ Exception processing chunk {i}: {str(e)}")
        continue

if len(embedding_rows) == 0:
    raise Exception("Failed to generate any embeddings. Check the endpoint and permissions.")

print(f"\n✓ Generated {len(embedding_rows)} embeddings successfully")
print(f"Embedding dimension: {len(embedding_rows[0].embedding)}\n")

# Create DataFrame with embeddings
embeddings_df = spark.createDataFrame(embedding_rows)

# Display sample
print("Sample embeddings (showing first 10 dimensions):")
embeddings_df.selectExpr(
    "document_name",
    "chunk_id",
    "substring(chunk_text, 1, 50) as chunk_preview",
    "slice(embedding, 1, 10) as embedding_preview"
).show(5, truncate=False)

# Save to table
embeddings_df.write.mode("overwrite").saveAsTable("hr_catalog.hr_core.hr_document_embeddings")

print(f"\n✓ Table 'hr_catalog.hr_core.hr_document_embeddings' created successfully!")
print(f"   - {len(embedding_rows)} embeddings saved")
print(f"   - Embedding dimension: {len(embedding_rows[0].embedding)}")

Generating embeddings for 8 chunks...

  ✓ Processed chunk 1/8: Security_Awareness_Brief_2025-01.pdf (chunk 1)
  ✓ Processed chunk 2/8: GDPR_Employee_Data_Practices_2025-01.pdf (chunk 1)
  ✓ Processed chunk 3/8: Exit_Process_Checklist_2025-01.pdf (chunk 1)
  ✓ Processed chunk 4/8: HR_Quarterly_Summary_2025Q1.pdf (chunk 1)
  ✓ Processed chunk 5/8: Travel_Policy_Quick_Reference_2025-01.pdf (chunk 1)
  ✓ Processed chunk 6/8: Performance_Calibration_Guide_2025-01.pdf (chunk 1)
  ✓ Processed chunk 7/8: DEI_Program_Outline_2025-01.pdf (chunk 1)
  ✓ Processed chunk 8/8: Flexible_Work_Toolkit_2025-01.pdf (chunk 1)

✓ Generated 8 embeddings successfully
Embedding dimension: 1024

Sample embeddings (showing first 10 dimensions):
+-----------------------------------------+--------+----------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
%sql
-- Verify the final embeddings table structure and content
SELECT 
  document_name,
  chunk_id,
  substring(chunk_text, 1, 80) AS chunk_preview,
  size(embedding) AS embedding_dimension,
  slice(embedding, 1, 5) AS embedding_sample
FROM hr_catalog.hr_core.hr_document_embeddings
ORDER BY document_name, chunk_id;

document_name,chunk_id,chunk_preview,embedding_dimension,embedding_sample
DEI_Program_Outline_2025-01.pdf,1,DEI Program Outline 2025-01 Owner: marcin.smith@nordstar.example.com (HR) Versi,1024,"List(0.024261474609375, -0.00780487060546875, -0.0316162109375, 0.02191162109375, -0.08306884765625)"
Exit_Process_Checklist_2025-01.pdf,1,Exit Process Checklist 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR,1024,"List(0.01352691650390625, -0.019775390625, 0.0010728836059570312, 0.025299072265625, -0.03363037109375)"
Flexible_Work_Toolkit_2025-01.pdf,1,Flexible Work Toolkit 2025-01 Owner: ava.white@nordstar.example.com (IT) Versio,1024,"List(0.04364013671875, 0.024444580078125, -0.01418304443359375, 0.0309906005859375, -0.0361328125)"
GDPR_Employee_Data_Practices_2025-01.pdf,1,GDPR Employee Data Practices 2025-01 Owner: noah.nowak@nordstar.example.com (HR,1024,"List(0.021636962890625, -0.004276275634765625, -0.0065460205078125, 0.0270538330078125, -0.049530029296875)"
HR_Quarterly_Summary_2025Q1.pdf,1,HR Quarterly Summary 2025Q1 Owner: sofia.wójcik@nordstar.example.com (HR) Versi,1024,"List(0.0295257568359375, 0.006168365478515625, -0.031524658203125, -0.00432586669921875, -0.04095458984375)"
Performance_Calibration_Guide_2025-01.pdf,1,Performance Calibration Guide 2025-01 Owner: olivia.rodriguez@nordstar.example.,1024,"List(0.007663726806640625, 0.00439453125, -0.0056915283203125, 0.027984619140625, -0.050048828125)"
Security_Awareness_Brief_2025-01.pdf,1,Security Awareness Brief 2025-01 Owner: marcin.dnbrowska@nordstar.example.com (,1024,"List(0.0213470458984375, 0.005096435546875, -0.00989532470703125, 0.03216552734375, -0.0528564453125)"
Travel_Policy_Quick_Reference_2025-01.pdf,1,Travel Policy Quick Reference 2025-01 Owner: lucas.williams@nordstar.example.co,1024,"List(0.0092010498046875, 0.0141754150390625, 0.041412353515625, 0.01470947265625, -0.033935546875)"


In [0]:
from pyspark.sql.functions import concat, col, lit

# Read the current embeddings table
embeddings_df = spark.table("hr_catalog.hr_core.hr_document_embeddings")

# Add ID column by concatenating document_name and chunk_id
embeddings_with_id = embeddings_df.withColumn(
    "id", 
    concat(col("document_name"), lit("_chunk_"), col("chunk_id").cast("string"))
).select(
    "id", "document_name", "chunk_id", "chunk_text", "embedding"
)

print(f"Recreating table with {embeddings_with_id.count()} rows...")

# Overwrite the table with the new structure
embeddings_with_id.write.mode("overwrite").saveAsTable("hr_catalog.hr_core.hr_document_embeddings")

# Change the id column to NOT NULL
spark.sql("""
    ALTER TABLE hr_catalog.hr_core.hr_document_embeddings
    ALTER COLUMN id SET NOT NULL
""")

# Now add the primary key constraint and enable CDF via SQL
spark.sql("""
    ALTER TABLE hr_catalog.hr_core.hr_document_embeddings
    ADD CONSTRAINT pk_embeddings PRIMARY KEY(id)
""")

spark.sql("""
    ALTER TABLE hr_catalog.hr_core.hr_document_embeddings 
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("\n✓ Table prepared for vector search:")
print("  - Primary key: id")
print("  - Change Data Feed: enabled")
print("  - Ready for Delta Sync index creation")

Recreating table with 8 rows...

✓ Table prepared for vector search:
  - Primary key: id
  - Change Data Feed: enabled
  - Ready for Delta Sync index creation


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import EndpointType
import time

w = WorkspaceClient()

# Configuration
endpoint_name = "hr-vector-search-endpoint"
index_name = "hr_catalog.hr_core.hr_document_embeddings_index"

print("Checking for existing vector search endpoints...\n")

# Check if endpoint exists
try:
    endpoint = w.vector_search_endpoints.get_endpoint(endpoint_name)
    print(f"✓ Endpoint '{endpoint_name}' already exists")
    print(f"  Status: {endpoint.endpoint_status.state}")
except Exception:
    # Endpoint doesn't exist, create it
    print(f"Creating new vector search endpoint '{endpoint_name}'...")
    print("  Type: STANDARD (optimized for low latency)\n")
    
    w.vector_search_endpoints.create_endpoint(
        name=endpoint_name,
        endpoint_type=EndpointType.STANDARD
    )
    
    print(f"✓ Endpoint '{endpoint_name}' created!")
    print("  Note: Endpoint provisioning takes a few minutes...\n")
    
    # Wait for endpoint to be ready
    print("Waiting for endpoint to be online...")
    for i in range(60):  # Wait up to 10 minutes
        try:
            endpoint = w.vector_search_endpoints.get_endpoint(endpoint_name)
            status = str(endpoint.endpoint_status.state)
            
            if "ONLINE" in status:
                print(f"\n✓ Endpoint is ONLINE and ready!")
                break
            else:
                print(f"  Status: {status}... ({i*10}s elapsed)")
                time.sleep(10)
        except Exception as e:
            print(f"  Waiting... ({i*10}s elapsed)")
            time.sleep(10)
    else:
        print("\n⚠ Endpoint is still provisioning. Continuing anyway...")

# Create the vector search index
print(f"\n{'='*60}")
print(f"Creating vector search index '{index_name}'...")
print(f"{'='*60}\n")

try:
    from databricks.sdk.service.vectorsearch import (
        DeltaSyncVectorIndexSpecRequest,
        EmbeddingVectorColumn,
        PipelineType,
        VectorIndexType
    )
    
    # Create Delta Sync index with self-managed embeddings
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=endpoint_name,
        primary_key="id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table="hr_catalog.hr_core.hr_document_embeddings",
            embedding_vector_columns=[
                EmbeddingVectorColumn(
                    name="embedding",
                    embedding_dimension=1024
                )
            ],
            pipeline_type=PipelineType.TRIGGERED
        )
    )
    
    print("\n✓ Vector search index created successfully!")
    print(f"\nIndex Details:")
    print(f"  - Index name: {index_name}")
    print(f"  - Endpoint: {endpoint_name}")
    print(f"  - Source table: hr_catalog.hr_core.hr_document_embeddings")
    print(f"  - Primary key: id")
    print(f"  - Embedding dimension: 1024")
    print(f"  - Pipeline type: TRIGGERED (sync manually)")
    print(f"\nNext steps:")
    print(f"  1. Sync the index: w.vector_search_indexes.sync_index(index_name='{index_name}')")
    print(f"  2. Query the index for semantic search on your HR documents")
    
except Exception as e:
    error_msg = str(e)
    if "already exists" in error_msg.lower():
        print(f"✓ Index '{index_name}' already exists")
        print("\nTo use it, trigger a sync:")
        print(f"  w.vector_search_indexes.sync_index(index_name='{index_name}')")
    else:
        print(f"\n✗ Error creating index: {e}")
        print("\nPossible issues:")
        print("  1. Endpoint is not ready yet (wait a few more minutes)")
        print("  2. Missing permissions on the endpoint or schema")
        print("  3. Primary key or Change Data Feed not enabled on the table")

Checking for existing vector search endpoints...

✓ Endpoint 'hr-vector-search-endpoint' already exists
  Status: EndpointStatusState.ONLINE

Creating vector search index 'hr_catalog.hr_core.hr_document_embeddings_index'...


✓ Vector search index created successfully!

Index Details:
  - Index name: hr_catalog.hr_core.hr_document_embeddings_index
  - Endpoint: hr-vector-search-endpoint
  - Source table: hr_catalog.hr_core.hr_document_embeddings
  - Primary key: id
  - Embedding dimension: 1024
  - Pipeline type: TRIGGERED (sync manually)

Next steps:
  1. Sync the index: w.vector_search_indexes.sync_index(index_name='hr_catalog.hr_core.hr_document_embeddings_index')
  2. Query the index for semantic search on your HR documents


In [0]:
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()

index_name = "hr_catalog.hr_core.hr_document_embeddings_index"
endpoint_name = "hr-vector-search-endpoint"

print("Checking vector search endpoint and index status...\n")
print("="*80)

# Check endpoint status first
try:
    endpoint = w.vector_search_endpoints.get_endpoint(endpoint_name)
    endpoint_status = str(endpoint.endpoint_status.state)
    print(f"Endpoint: {endpoint_name}")
    print(f"Status: {endpoint_status}\n")
    
    if "ONLINE" not in endpoint_status:
        print("⚠ Endpoint is not online yet.")
        print("  The endpoint must be fully provisioned before the index can be synced.")
        print("  This typically takes 5-10 minutes after creation.")
        print(f"\n  Current status: {endpoint_status}")
        print("\n  Please wait and re-run this cell when the endpoint is ONLINE.")
    else:
        print("✓ Endpoint is ONLINE!\n")
        print("-"*80)
        
        # Check index status
        index_info = w.vector_search_indexes.get_index(index_name=index_name)
        print(f"\nIndex: {index_name}")
        print(f"Ready: {index_info.status.ready}")
        print(f"Message: {index_info.status.message}\n")
        
        if index_info.status.ready:
            print("✓ Index is already synced and ready for queries!")
            print(f"  Indexed rows: {index_info.status.indexed_row_count}")
        else:
            print("Triggering index sync...\n")
            
            try:
                w.vector_search_indexes.sync_index(index_name=index_name)
                print("✓ Sync triggered successfully!")
                print("\nWaiting for index to be ready...")
                
                # Wait for index to be ready
                for i in range(30):  # Wait up to 5 minutes
                    time.sleep(10)
                    index_info = w.vector_search_indexes.get_index(index_name=index_name)
                    
                    if index_info.status.ready:
                        print(f"\n✓ Index is ready!")
                        print(f"  Indexed rows: {index_info.status.indexed_row_count}")
                        print("\n✓ You can now run Cell 13 to query the index!")
                        break
                    else:
                        print(f"  Syncing... ({(i+1)*10}s elapsed)")
                else:
                    print("\n⚠ Index is still syncing. Please wait a few more minutes and check again.")
                    
            except Exception as e:
                if "not ready" in str(e).lower():
                    print("⚠ Index sync cannot start yet - endpoint is still provisioning.")
                    print("   Please re-run this cell in a few minutes.")
                else:
                    raise
                    
except Exception as e:
    print(f"✗ Error: {e}")
    
print("\n" + "="*80)

Checking vector search endpoint and index status...

Endpoint: hr-vector-search-endpoint
Status: EndpointStatusState.ONLINE

✓ Endpoint is ONLINE!

--------------------------------------------------------------------------------

Index: hr_catalog.hr_core.hr_document_embeddings_index
Ready: False
Message: Delta sync index creation is pending endpoint provisioning.

Triggering index sync...

⚠ Index sync cannot start yet - endpoint is still provisioning.
   Please re-run this cell in a few minutes.



In [0]:
from databricks.vector_search.client import VectorSearchClient
import requests
import json

# Initialize vector search client
vsc = VectorSearchClient()

# Configuration
index_name = "hr_catalog.hr_core.hr_document_embeddings_index"

print("Loading vector search index...\n")

# Get the vector search index
vector_index = vsc.get_index(
    endpoint_name="hr-vector-search-endpoint",
    index_name=index_name
)

print("✓ Vector Search Index loaded successfully!\n")

# Setup embedding endpoint
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()
host = ctx.apiUrl().get()

url = f"{host}/serving-endpoints/databricks-bge-large-en/invocations"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

# Define all questions to ask
questions = [
    "What are the GDPR requirements for employee data?",
    "What expenses are covered under the travel policy?",
    "What are the required security awareness activities?",
    "How does the performance calibration process work?",
    "What steps must be completed during employee exit?"
]

# Store all results
all_query_results = {}

print("="*80)
print(f"Processing {len(questions)} questions...")
print("="*80)

for idx, query in enumerate(questions, 1):
    print(f"\n\n{'='*80}")
    print(f"QUESTION {idx}/{len(questions)}")
    print("="*80)
    print(f"Query: {query}\n")
    
    try:
        # Generate embedding for the query
        payload = {"input": [query]}
        response = requests.post(url, headers=headers, json=payload)
        query_embedding = response.json()['data'][0]['embedding']
        
        print(f"✓ Generated query embedding (dimension: {len(query_embedding)})")
        
        # Search the vector index
        results = vector_index.similarity_search(
            query_vector=query_embedding,
            columns=["document_name", "chunk_id", "chunk_text"],
            num_results=3
        )
        
        # Store results for later use
        all_query_results[query] = results
        
        print(f"✓ Found {len(results['result']['data_array'])} relevant chunks\n")
        print("-"*80)
        
        # Display results
        for i, row in enumerate(results['result']['data_array'], 1):
            doc_name = row[0]
            chunk_id = row[1]
            chunk_text = row[2]
            
            print(f"\nResult {i}:")
            print(f"  Document: {doc_name}")
            print(f"  Chunk ID: {chunk_id}")
            print(f"  Text: {chunk_text[:150]}..." if len(chunk_text) > 150 else f"  Text: {chunk_text}")
            
    except Exception as e:
        if "not ready" in str(e).lower():
            print(f"\n⚠ Index is not ready yet. Please run Cell 12 to sync the index first.")
            print(f"   (This typically takes 5-10 minutes after endpoint provisioning)")
        else:
            print(f"\n✗ Error: {e}")
        raise

print(f"\n\n{'='*80}")
print(f"✓ Successfully processed all {len(questions)} questions!")
print("="*80)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Loading vector search index...

✓ Vector Search Index loaded successfully!

Processing 5 questions...


QUESTION 1/5
Query: What are the GDPR requirements for employee data?

✓ Generated query embedding (dimension: 1024)
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
✓ Found 3 relevant chunks

--------------------------------------------------------------------------------

Result 1:
  Document: GDPR_Employee_Data_Practices_2025-01.pdf
  Chunk ID: 1.0
  Text: GDPR Employee Data Practices 2025-01

Owner: noah.nowak@nordstar.example.com (HR)
Version: 2025-01
• Lawful bases overview
• DSAR intake ® response
• ...

Result 2

In [0]:
# RAG Prompt Template
RAG_PROMPT_TEMPLATE = """Answer the question using only the supplied context.

If the answer is not found, say that the information is not available.

Include citations.

Context:
{retrieved_chunks}

Question:
{user_question}
"""

print("="*80)
print("BUILDING RAG PROMPTS FOR ALL QUESTIONS")
print("="*80)

# Build RAG prompts for each question
for idx, (user_question, results) in enumerate(all_query_results.items(), 1):
    print(f"\n\n{'#'*80}")
    print(f"QUESTION {idx}: {user_question}")
    print("#"*80)
    
    # Format the retrieved chunks
    retrieved_chunks_text = ""
    for i, row in enumerate(results['result']['data_array'], 1):
        doc_name = row[0]
        chunk_id = row[1]
        chunk_text = row[2]
        retrieved_chunks_text += f"[{i}] Source: {doc_name} (chunk {chunk_id})\n{chunk_text}\n\n"
    
    # Build the complete prompt
    final_prompt = RAG_PROMPT_TEMPLATE.format(
        retrieved_chunks=retrieved_chunks_text.strip(),
        user_question=user_question
    )
    
    print("\n" + "-"*80)
    print("RAG PROMPT:")
    print("-"*80)
    print(final_prompt)
    print("-"*80)

print(f"\n\n{'='*80}")
print(f"✓ Successfully built RAG prompts for all {len(all_query_results)} questions!")
print("="*80)
print("\nNext steps:")
print("  1. Send each prompt to your LLM (e.g., OpenAI, Databricks Foundation Models)")
print("  2. The LLM will generate answers using only the retrieved context")
print("  3. Citations will reference the document sources")

BUILDING RAG PROMPTS FOR ALL QUESTIONS


################################################################################
QUESTION 1: What are the GDPR requirements for employee data?
################################################################################

--------------------------------------------------------------------------------
RAG PROMPT:
--------------------------------------------------------------------------------
Answer the question using only the supplied context.

If the answer is not found, say that the information is not available.

Include citations.

Context:
[1] Source: GDPR_Employee_Data_Practices_2025-01.pdf (chunk 1.0)
GDPR Employee Data Practices 2025-01

Owner: noah.nowak@nordstar.example.com (HR)
Version: 2025-01
• Lawful bases overview
• DSAR intake ® response
• Retention map snapshot
• Vendors & DPAs
• Cross-border safeguards

[2] Source: HR_Quarterly_Summary_2025Q1.pdf (chunk 1.0)
HR Quarterly Summary 2025Q1

Owner: sofia.wójcik@nordstar.example.c

In [0]:
from pyspark.sql import Row
from datetime import datetime

# Flatten all query results into rows
export_rows = []

for question, results in all_query_results.items():
    # Extract the result chunks
    data_array = results['result']['data_array']
    
    # Create a row for each retrieved chunk
    for rank, row in enumerate(data_array, 1):
        doc_name = row[0]
        chunk_id = row[1]
        chunk_text = row[2]
        
        export_rows.append(
            Row(
                query_timestamp=datetime.now(),
                question=question,
                rank=rank,
                document_name=doc_name,
                chunk_id=int(chunk_id),
                chunk_text=chunk_text,
                chunk_length=len(chunk_text)
            )
        )

print(f"Exporting {len(export_rows)} result records...\n")

# Create DataFrame
results_df = spark.createDataFrame(export_rows)

# Show preview
print("Preview of export data:")
results_df.select(
    "question", "rank", "document_name", "chunk_id"
).show(15, truncate=False)

# Save to table
table_name = "hr_catalog.hr_core.hr_rag_query_results"
results_df.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Results exported to table: {table_name}")
print(f"   - Total rows: {len(export_rows)}")
print(f"   - Questions: {len(all_query_results)}")
print(f"   - Top results per question: 3")

print("\nTable schema:")
print("  - query_timestamp: timestamp of the query")
print("  - question: the user's question")
print("  - rank: result ranking (1-3)")
print("  - document_name: source document")
print("  - chunk_id: chunk identifier")
print("  - chunk_text: retrieved text chunk")
print("  - chunk_length: length of chunk text")

Exporting 15 result records...

Preview of export data:
+----------------------------------------------------+----+-----------------------------------------+--------+
|question                                            |rank|document_name                            |chunk_id|
+----------------------------------------------------+----+-----------------------------------------+--------+
|What are the GDPR requirements for employee data?   |1   |GDPR_Employee_Data_Practices_2025-01.pdf |1       |
|What are the GDPR requirements for employee data?   |2   |HR_Quarterly_Summary_2025Q1.pdf          |1       |
|What are the GDPR requirements for employee data?   |3   |Exit_Process_Checklist_2025-01.pdf       |1       |
|What expenses are covered under the travel policy?  |1   |Travel_Policy_Quick_Reference_2025-01.pdf|1       |
|What expenses are covered under the travel policy?  |2   |Exit_Process_Checklist_2025-01.pdf       |1       |
|What expenses are covered under the travel policy?  |3 

In [0]:
%sql
-- View summary of exported RAG query results
SELECT 
  question,
  rank,
  document_name,
  chunk_id,
  chunk_length,
  substring(chunk_text, 1, 100) as chunk_preview
FROM hr_catalog.hr_core.hr_rag_query_results
ORDER BY question, rank;

question,rank,document_name,chunk_id,chunk_length,chunk_preview
How does the performance calibration process work?,1,Performance_Calibration_Guide_2025-01.pdf,1,200,Performance Calibration Guide 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 20
How does the performance calibration process work?,2,DEI_Program_Outline_2025-01.pdf,1,196,DEI Program Outline 2025-01 Owner: marcin.smith@nordstar.example.com (HR) Version: 2025-01 • ERG la
How does the performance calibration process work?,3,GDPR_Employee_Data_Practices_2025-01.pdf,1,215,GDPR Employee Data Practices 2025-01 Owner: noah.nowak@nordstar.example.com (HR) Version: 2025-01 •
What are the GDPR requirements for employee data?,1,GDPR_Employee_Data_Practices_2025-01.pdf,1,215,GDPR Employee Data Practices 2025-01 Owner: noah.nowak@nordstar.example.com (HR) Version: 2025-01 •
What are the GDPR requirements for employee data?,2,HR_Quarterly_Summary_2025Q1.pdf,1,231,HR Quarterly Summary 2025Q1 Owner: sofia.wójcik@nordstar.example.com (HR) Version: 2025-01 • Headco
What are the GDPR requirements for employee data?,3,Exit_Process_Checklist_2025-01.pdf,1,213,Exit Process Checklist 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 2025-01 •
What are the required security awareness activities?,1,Security_Awareness_Brief_2025-01.pdf,1,227,Security Awareness Brief 2025-01 Owner: marcin.dnbrowska@nordstar.example.com (IT) Version: 2025-01
What are the required security awareness activities?,2,GDPR_Employee_Data_Practices_2025-01.pdf,1,215,GDPR Employee Data Practices 2025-01 Owner: noah.nowak@nordstar.example.com (HR) Version: 2025-01 •
What are the required security awareness activities?,3,Exit_Process_Checklist_2025-01.pdf,1,213,Exit Process Checklist 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 2025-01 •
What expenses are covered under the travel policy?,1,Travel_Policy_Quick_Reference_2025-01.pdf,1,233,Travel Policy Quick Reference 2025-01 Owner: lucas.williams@nordstar.example.com (Finance) Version:
